In [1]:
# Cellule 1: importe les bibliotheques et configure Altair.
import altair as alt
import pandas as pd
import geopandas as gpd # Requires geopandas -- e.g.: conda install -c conda-forge geopandas
alt.data_transformers.enable('default', max_rows=None) # Inline data so static rendering can access it
alt.renderers.enable('png') # Static renderer: avoids frontend vega-embed/requirejs issues in VS Code notebooks
import ipywidgets as widgets
from IPython.display import clear_output, display
from pathlib import Path

In [2]:
# Cellule 2: charge et prepare les donnees
data_path = Path('..') / 'dpt2020.csv'
just_names = pd.read_csv(data_path, sep=';')
just_names = just_names[just_names['preusuel'] != '_PRENOMS_RARES']
just_names = just_names[just_names['dpt'] != 'XX']

years = sorted(just_names['annais'].astype(str).str.strip().astype(int).unique())

# Fixed color map for all first names (stable across years)
all_names = sorted(just_names['preusuel'].dropna().astype(str).unique())
base_colors = [
    '#4C78A8', '#F58518', '#E45756', '#72B7B2', '#54A24B',
    '#EECA3B', '#B279A2', '#FF9DA6', '#9D755D', '#BAB0AC',
    '#1F77B4', '#FF7F0E', '#2CA02C', '#D62728', '#9467BD',
    '#8C564B', '#E377C2', '#7F7F7F', '#BCBD22', '#17BECF',
    '#264653', '#2A9D8F', '#E9C46A', '#F4A261', '#E76F51',
    '#3A86FF', '#8338EC', '#FF006E', '#FB5607', '#06D6A0',
    '#118AB2', '#073B4C', '#8E9AAF', '#CBC0D3', '#EFD3D7'
]
color_range = [base_colors[i % len(base_colors)] for i in range(len(all_names))]

# Fixed x-axis scale for all years: global max + 5000
max_nombre_global = (
    just_names
    .groupby(['annais', 'preusuel'], as_index=False)['nombre']
    .sum()['nombre']
    .max()
)
x_max = int(max_nombre_global) + 5000

# Precompute shared aggregates once
yearly_name_totals = (
    just_names.assign(annais_clean=just_names['annais'].astype(str).str.strip().astype(int))
    .groupby(['annais_clean', 'preusuel'], as_index=False)['nombre']
    .sum()
)
annual_stats = (
    yearly_name_totals
    .groupby('annais_clean', as_index=False)
    .agg(
        births_this_year=('nombre', 'sum'),
        names_over_1000=('nombre', lambda series: int((series > 1000).sum()))
    )
    .set_index('annais_clean')
)

# Precompute yearly full ranking once to measure rank movement
ranks_by_year = {}
for year in years:
    year_ranking = (
        yearly_name_totals[yearly_name_totals['annais_clean'] == year]
        .sort_values(['nombre', 'preusuel'], ascending=[False, True])
        .reset_index(drop=True)
    )
    year_ranking['rank'] = year_ranking.index + 1
    ranks_by_year[year] = dict(zip(year_ranking['preusuel'], year_ranking['rank']))

def trend_label(delta):
    if pd.isna(delta) or int(delta) == 0:
        return ''
    delta_int = int(delta)
    return f'↑ {delta_int}' if delta_int > 0 else f'↓ {abs(delta_int)}'

def trend_color(delta):
    if pd.isna(delta) or int(delta) == 0:
        return '#6b7280'
    return '#16a34a' if int(delta) > 0 else '#dc2626'

In [ ]:
# Cellule 3: cree les widgets de controle (slider annee, vitesse, play, mode d affichage).
annee_slider = widgets.IntSlider(
    value=1900,
    min=years[0],
    max=years[-1],
    step=1,
    description='Année',
    continuous_update=False,
    readout=True,
    readout_format='d',
    tooltip='Choisir l année affichée'
)
annee_slider.style = {'description_width': '48px'}
annee_slider.layout = widgets.Layout(width='190px', margin='0px')
annee_input = widgets.BoundedIntText(
    value=annee_slider.value,
    min=annee_slider.min,
    max=annee_slider.max,
    step=1,
    description='',
    layout=widgets.Layout(width='72px', margin='0px')
)
widgets.jslink((annee_slider, 'value'), (annee_input, 'value'))
annee_slider.readout = False

vitesse_slider = widgets.FloatSlider(
    value=2.0,
    min=0.5,
    max=5.0,
    step=0.1,
    description='Durée (s)',
    continuous_update=False,
    readout=True,
    readout_format='.1f',
    tooltip='Durée en secondes par année (plus grand = plus lent)'
)
vitesse_slider.style = {'description_width': '58px'}
vitesse_slider.layout = widgets.Layout(width='250px', margin='0px')

display_mode_toggle = widgets.Dropdown(
    options=[('Brut', 'raw'), ('%', 'percent')],
    value='raw',
    description='Affichage',
    tooltip='Choisir valeurs brutes ou pourcentages',
    layout=widgets.Layout(width='170px', min_width='170px')
)
display_mode_toggle.style = {'description_width': '70px'}

btn_play = widgets.ToggleButton(
    value=False,
    description='',
    icon='play',
    tooltip='Lancer le défilement des années',
    layout=widgets.Layout(width='36px', height='30px')
)


play_engine = widgets.Play(
    value=annee_slider.value,
    min=annee_slider.min,
    max=annee_slider.max,
    step=1,
    interval=int(vitesse_slider.value * 1000),
    description=''
)

# Keep Play rendered (tiny/invisible) so ticks are reliable in VS Code.
play_engine.layout = widgets.Layout(width='1px', height='1px', opacity='0', overflow='hidden')

top_card_widget = widgets.HTML()
stats_left_widget = widgets.HTML()
stats_right_widget = widgets.HTML()
chart_widget = widgets.Image(format='png', layout=widgets.Layout(width='700px'))

In [4]:
# Cellule 4: construit les fonctions de rendu et le cache lazy des cartes/graphes.
def stat_html(label, value):
    return f'''<div style="width:340px;height:60px;background:#e5e7eb;border-radius:6px;display:flex;flex-direction:column;justify-content:center;align-items:center;font-family:sans-serif;color:#1f2937;">
<div style="font-size:10px;color:#4b5563;line-height:1.05;">{label}</div>
<div style="font-size:16px;font-weight:700;line-height:1.05;">{value}</div>
</div>'''

def build_chart(top10_annee, annee, mode, births_this_year):
    top10_plot = top10_annee.copy()
    if mode == 'percent':
        top10_plot['metric_value'] = top10_plot['pct_births']
        top10_plot['metric_label'] = top10_plot['pct_label']
        top10_plot['metric_trend'] = top10_plot['value_trend_pct']
        top10_plot['metric_trend_color'] = top10_plot['value_trend_color_pct']
        x_title = 'Part des naissances (%)'
        x_scale = alt.Scale(domain=[0, 13])
    else:
        top10_plot['metric_value'] = top10_plot['nombre']
        top10_plot['metric_label'] = top10_plot['nombre_label']
        top10_plot['metric_trend'] = top10_plot['value_trend']
        top10_plot['metric_trend_color'] = top10_plot['value_trend_color']
        x_title = 'Nombre de naissances'
        x_scale = alt.Scale(domain=[0, x_max])

    if annee == years[0]:
        top10_plot['metric_trend'] = ''
    name_order = top10_plot['preusuel'].tolist()

    bars_mark = alt.Chart(top10_plot).mark_bar().encode(
        x=alt.X(
            'metric_value:Q',
            title=x_title,
            axis=alt.Axis(domain=False),
            scale=x_scale
        ),
        y=alt.Y(
            'preusuel:N',
            sort=name_order,
            title=None,
            axis=alt.Axis(domain=False, labels=False, ticks=False)
        ),
        color=alt.Color(
            'preusuel:N',
            legend=None,
            scale=alt.Scale(domain=all_names, range=color_range)
        ),
        tooltip=['preusuel:N', 'nombre:Q', alt.Tooltip('pct_births:Q', title='% naissances', format='.2f'), 'prev_nombre:Q', 'nombre_delta:Q', alt.Tooltip('pct_delta:Q', title='Delta %', format='.2f'), 'rank:Q', 'prev_rank:Q', 'rank_delta:Q']
    )

    values_mark = alt.Chart(top10_plot).mark_text(
        align='left',
        baseline='middle',
        dx=5
    ).encode(
        x='metric_value:Q',
        y=alt.Y(
            'preusuel:N',
            sort=name_order,
            axis=alt.Axis(domain=False)
        ),
        text='metric_label:N'
    )

    values_trend_mark = alt.Chart(top10_plot[top10_plot['metric_trend'] != '']).mark_text(
        align='left',
        baseline='middle',
        fontWeight='bold',
        dx=42
    ).encode(
        x='metric_value:Q',
        y=alt.Y(
            'preusuel:N',
            sort=name_order,
            axis=alt.Axis(domain=False)
        ),
        text='metric_trend:N',
        color=alt.Color('metric_trend_color:N', scale=None, legend=None)
    )

    names_mark = alt.Chart(top10_plot).mark_text(
        align='right',
        baseline='middle',
        dx=-48,
        color='#1f2937'
    ).encode(
        x=alt.value(0),
        y=alt.Y(
            'preusuel:N',
            sort=name_order,
            axis=alt.Axis(domain=False)
        ),
        text='preusuel:N'
    )

    trend_mark = alt.Chart(top10_plot[top10_plot['trend'] != '']).mark_text(
        align='right',
        baseline='middle',
        fontWeight='bold',
        dx=-18
    ).encode(
        x=alt.value(0),
        y=alt.Y(
            'preusuel:N',
            sort=name_order,
            axis=alt.Axis(domain=False)
        ),
        text='trend:N',
        color=alt.Color('trend_color:N', scale=None, legend=None)
    )

    bars = bars_mark + values_mark + values_trend_mark + names_mark + trend_mark
    return bars.properties(
        title=f'Top 10 des prénoms en {annee}',
        width=700,
        height=350
    )

# Lazy caches: keep existing cache on rerun to avoid rebuilding.
if 'yearly_payload' not in globals():
    yearly_payload = {}
if 'cards_by_year' not in globals():
    cards_by_year = {}
if 'charts_by_year' not in globals():
    charts_by_year = {}

def get_year_payload(annee):
    if annee in yearly_payload:
        cached_payload = yearly_payload[annee]
        required_cols = {'prev_nombre', 'nombre_delta', 'value_trend', 'value_trend_color','pct_births', 'pct_delta', 'value_trend_pct', 'value_trend_color_pct', 'pct_label', 'nombre_label'}
        if required_cols.issubset(set(cached_payload['top10'].columns)):
            return cached_payload
        # Invalidate stale cache entries created before value-trend fields existed.
        yearly_payload.pop(annee, None)
        cards_by_year.pop(annee, None)
        charts_by_year.pop((annee, 'raw'), None)
        charts_by_year.pop((annee, 'percent'), None)

    year_data = yearly_name_totals[yearly_name_totals['annais_clean'] == annee]
    top10_annee = (
        year_data
        .drop(columns=['annais_clean'])
        .sort_values('nombre', ascending=False)
        .head(10)
        .copy()
    )

    current_ranks = ranks_by_year[annee]
    previous_ranks = ranks_by_year.get(annee - 1, {})
    top10_annee['rank'] = top10_annee['preusuel'].map(current_ranks)
    top10_annee['prev_rank'] = top10_annee['preusuel'].map(previous_ranks)
    top10_annee['rank_delta'] = top10_annee['prev_rank'] - top10_annee['rank']
    top10_annee['trend'] = top10_annee['rank_delta'].apply(trend_label)
    top10_annee['trend_color'] = top10_annee['rank_delta'].apply(trend_color)

    previous_values = (
        yearly_name_totals[yearly_name_totals['annais_clean'] == annee - 1]
        .set_index('preusuel')['nombre']
        .to_dict()
    )
    top10_annee['prev_nombre'] = top10_annee['preusuel'].map(previous_values).fillna(0)
    top10_annee['nombre_delta'] = top10_annee['nombre'] - top10_annee['prev_nombre']

    births_this_year = int(annual_stats.loc[annee, 'births_this_year'])
    prev_births_this_year = int(annual_stats.loc[annee - 1, 'births_this_year']) if (annee - 1) in annual_stats.index else 0

    top10_annee['pct_births'] = top10_annee['nombre'].apply(
        lambda n: (float(n) * 100.0 / births_this_year) if births_this_year else 0.0
    )
    top10_annee['prev_pct_births'] = top10_annee['prev_nombre'].apply(
        lambda n: (float(n) * 100.0 / prev_births_this_year) if prev_births_this_year else 0.0
    )
    top10_annee['pct_delta'] = top10_annee['pct_births'] - top10_annee['prev_pct_births']

    def value_trend_label(delta):
        delta_int = int(delta)
        if delta_int > 0:
            return '↑'
        if delta_int < 0:
            return '↓'
        return ''

    def value_trend_color(delta):
        if int(delta) > 0:
            return '#16a34a'
        if int(delta) < 0:
            return '#dc2626'
        return '#6b7280'

    def value_trend_label_pct(delta):
        if delta > 0:
            return '↑'
        if delta < 0:
            return '↓'
        return ''

    def value_trend_color_pct(delta):
        if delta > 0:
            return '#16a34a'
        if delta < 0:
            return '#dc2626'
        return '#6b7280'

    top10_annee['value_trend'] = top10_annee['nombre_delta'].apply(value_trend_label)
    top10_annee['value_trend_color'] = top10_annee['nombre_delta'].apply(value_trend_color)
    top10_annee['value_trend_pct'] = top10_annee['pct_delta'].apply(value_trend_label_pct)
    top10_annee['value_trend_color_pct'] = top10_annee['pct_delta'].apply(value_trend_color_pct)

    top10_annee['nombre_label'] = top10_annee['nombre'].astype(int).map(lambda v: f'{v:,}'.replace(',', ' '))
    top10_annee['pct_label'] = top10_annee['pct_births'].map(lambda v: f'{v:.2f}%')

    if annee == years[0]:
        top10_annee['value_trend'] = ''
        top10_annee['value_trend_pct'] = ''

    names_over_1000 = int(annual_stats.loc[annee, 'names_over_1000'])
    top1_name = top10_annee.iloc[0]['preusuel'] if len(top10_annee) else ''

    payload = {
        'top10': top10_annee,
        'births_this_year': births_this_year,
        'names_over_1000': names_over_1000,
        'top1_name': top1_name
    }
    yearly_payload[annee] = payload
    return payload

def get_card_and_chart(annee, mode):
    chart_key = (annee, mode)
    if annee in cards_by_year and chart_key in charts_by_year:
        return cards_by_year[annee], charts_by_year[chart_key]

    payload = get_year_payload(annee)
    top10_annee = payload['top10']
    births_this_year = payload['births_this_year']
    names_over_1000 = payload['names_over_1000']
    top1_name = payload['top1_name']

    top_card = f'''<div style="width:700px;height:80px;background:#ffd166;border-radius:0px;display:flex;justify-content:center;align-items:center;font-family:sans-serif;color:#1f2937;font-size:22px;font-weight:700;">
Prénom de l'année : {top1_name}
</div>'''
    stats_left = stat_html('Naissances cette année', f"{births_this_year:,}".replace(',', ' '))
    stats_right = stat_html('Prénoms > 1000 naissances', str(names_over_1000))

    cards_by_year[annee] = (top_card, stats_left, stats_right)
    charts_by_year[chart_key] = build_chart(top10_annee, annee, mode, births_this_year)
    return cards_by_year[annee], charts_by_year[chart_key]

In [21]:
# Cellule 5: definit les callbacks et met en page le tableau de bord.
last_render_key = None

# Ensure chart widget exists even if this cell is run standalone.
if 'chart_widget' not in globals():
    chart_widget = widgets.Image(format='png', layout=widgets.Layout(width='700px'))

def render_all(force=False):
    global last_render_key
    annee = annee_slider.value
    mode = display_mode_toggle.value
    current_key = (annee, mode)

    # Skip redraw if nothing visible changed.
    if not force and last_render_key == current_key:
        return

    (top_card, stats_left, stats_right), bars = get_card_and_chart(annee, mode)

    top_card_widget.value = top_card
    stats_left_widget.value = stats_left
    stats_right_widget.value = stats_right

    # Render chart as PNG bytes for robust single-display behavior in VS Code notebooks.
    import io
    png_buffer = io.BytesIO()
    bars.save(png_buffer, format='png')
    chart_widget.value = png_buffer.getvalue()

    last_render_key = current_key

def refresh_chart(_change=None):
    render_all()

def update_play_interval(_change):
    play_engine.interval = int(vitesse_slider.value * 1000)

def on_play_step(change):
    if change['name'] != 'value':
        return
    # Ignore buffered ticks when paused.
    if not play_engine.playing:
        return
    new_year = int(change['new'])
    if annee_slider.value != new_year:
        annee_slider.value = new_year

def on_play_toggle(change):
    if change['name'] != 'value':
        return

    if change['new'] and annee_slider.value >= annee_slider.max:
        annee_slider.value = annee_slider.min
        play_engine.value = annee_slider.value

    if not change['new']:
        # Hard-stop play and lock value to current year to prevent post-pause drift.
        play_engine.playing = False
        play_engine.value = annee_slider.value

    btn_play.icon = 'pause' if change['new'] else 'play'
    btn_play.tooltip = (
        'Arrêter le défilement des années'
        if change['new']
        else 'Lancer le défilement des années'
    )

# Center cards and chart block on the same visual width.
stats_box = widgets.HBox(
    [stats_left_widget, stats_right_widget],
    layout=widgets.Layout(width='700px', justify_content='space-between', gap='16px')
)

# Add a small white margin around the chart.
chart_box = widgets.Box(
    [chart_widget],
    layout=widgets.Layout(width='724px', padding='12px', margin='0px')
)


In [22]:
# Cellule 6: relie les widgets, reinitialise l'etat et affiche l'interface.
# Cette cellule est autonome pour redemarrer la visualisation.
import inspect

annee_slider = widgets.IntSlider(
    value=1900,
    min=years[0],
    max=years[-1],
    step=1,
    description='Année',
    continuous_update=False,
    readout=True,
    readout_format='d',
    tooltip='Choisir l année affichée'
)
annee_slider.style = {'description_width': '48px'}
annee_slider.layout = widgets.Layout(width='190px', margin='0px')
annee_input = widgets.BoundedIntText(
    value=annee_slider.value,
    min=annee_slider.min,
    max=annee_slider.max,
    step=1,
    description='',
    layout=widgets.Layout(width='72px', margin='0px')
)
widgets.jslink((annee_slider, 'value'), (annee_input, 'value'))
annee_slider.readout = False

vitesse_slider = widgets.FloatSlider(
    value=2.0,
    min=0.5,
    max=5.0,
    step=0.1,
    description='Durée (s)',
    continuous_update=False,
    readout=True,
    readout_format='.1f',
    tooltip='Durée en secondes par année (plus grand = plus lent)'
)
vitesse_slider.style = {'description_width': '58px'}
vitesse_slider.layout = widgets.Layout(width='250px', margin='0px')

display_mode_toggle = widgets.Dropdown(
    options=[('Brut', 'raw'), ('%', 'percent')],
    value='raw',
    description='Affichage',
    tooltip='Choisir valeurs brutes ou pourcentages',
    layout=widgets.Layout(width='170px', min_width='170px')
)
display_mode_toggle.style = {'description_width': '70px'}

btn_play = widgets.ToggleButton(
    value=False,
    description='',
    icon='play',
    tooltip='Lancer le défilement des années',
    layout=widgets.Layout(width='36px', height='30px')
)

play_engine = widgets.Play(
    value=annee_slider.value,
    min=annee_slider.min,
    max=annee_slider.max,
    step=1,
    interval=int(vitesse_slider.value * 1000),
    description=''
)

# Keep Play rendered (tiny/invisible) so ticks are reliable in VS Code.
play_engine.layout = widgets.Layout(width='1px', height='1px', opacity='0', overflow='hidden')

top_card_widget = widgets.HTML()
stats_left_widget = widgets.HTML()
stats_right_widget = widgets.HTML()
chart_widget = widgets.Image(format='png', layout=widgets.Layout(width='700px'))

# Center cards and chart block on the same visual width.
stats_box = widgets.HBox(
    [stats_left_widget, stats_right_widget],
    layout=widgets.Layout(width='700px', justify_content='space-between', gap='16px')
)

# Add a small white margin around the chart.
chart_box = widgets.Box(
    [chart_widget],
    layout=widgets.Layout(width='724px', padding='12px', margin='0px')
)

play_btn_link = widgets.jslink((btn_play, 'value'), (play_engine, 'playing'))

# Backward compatibility: if an old kernel still has get_card_and_chart(annee),
# wrap it so render_all() can always call get_card_and_chart(annee, mode).
signature_params = list(inspect.signature(get_card_and_chart).parameters)
if len(signature_params) == 1:
    _old_get_card_and_chart = get_card_and_chart
    def get_card_and_chart(annee, mode='raw'):
        return _old_get_card_and_chart(annee)

# Force fresh chart/card cache so new arrow logic is always applied.
if 'cards_by_year' in globals():
    cards_by_year.clear()
if 'charts_by_year' in globals():
    charts_by_year.clear()
last_render_key = None

# Wire callbacks.
annee_slider.observe(refresh_chart, names='value')
vitesse_slider.observe(update_play_interval, names='value')
display_mode_toggle.observe(refresh_chart, names='value')
btn_play.observe(on_play_toggle, names='value')
play_engine.observe(on_play_step, names='value')

# Reset controls on launch.
btn_play.value = False
play_engine.playing = False
display_mode_toggle.value = 'raw'
annee_slider.value = annee_slider.min
play_engine.value = annee_slider.value
update_play_interval(None)

render_all(force=True)
timeline_controls = widgets.HBox(
    [btn_play, annee_slider, annee_input, vitesse_slider],
    layout=widgets.Layout(gap='1px', align_items='center')
)
# Use a fixed spacer because layout gap can be ignored in VS Code widget renderer.
display_spacer = widgets.Box(layout=widgets.Layout(width='50px', min_width='50px'))
controls_row = widgets.HBox(
    [display_mode_toggle, display_spacer, timeline_controls],
    layout=widgets.Layout(align_items='center')
)
display(widgets.VBox([
    top_card_widget,
    stats_box,
    chart_box,
    controls_row,
    play_engine
], layout=widgets.Layout(align_items='center')))
